In [2]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [4]:
from sklearn.linear_model import Ridge

In [5]:
file_path = "data/cleaned_debris2.csv"
df = pd.read_csv(file_path)

In [6]:
print("Shape:", df.shape)
print(df.head())
print(df.tail())

Shape: (816, 8)
      Date  Spacecraft  Rocket Bodies  Mission-Related Debris  \
0  1957.00         0.0            0.0                     0.0   
1  1957.09         0.0            0.0                     0.0   
2  1957.16         0.0            0.0                     0.0   
3  1957.25         0.0            0.0                     0.0   
4  1957.33         0.0            0.0                     0.0   

   Fragmentation Debris  Unassigned Type  Total  Year  
0                   0.0              0.0    0.0  1957  
1                   0.0              0.0    0.0  1957  
2                   0.0              0.0    0.0  1957  
3                   0.0              0.0    0.0  1957  
4                   0.0              0.0    0.0  1957  
        Date  Spacecraft  Rocket Bodies  Mission-Related Debris  \
811  2024.58     13032.0         2128.0                  1888.0   
812  2024.67     13046.0         2124.0                  1895.0   
813  2024.75     13151.0         2129.0                 

In [7]:
print(df.isnull().sum())

Date                      0
Spacecraft                0
Rocket Bodies             0
Mission-Related Debris    0
Fragmentation Debris      0
Unassigned Type           0
Total                     0
Year                      0
dtype: int64


In [8]:
print("Total null values:", df.isnull().sum().sum())

Total null values: 0


In [9]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [10]:
print(df.dtypes)

Date                      float64
Spacecraft                float64
Rocket Bodies             float64
Mission-Related Debris    float64
Fragmentation Debris      float64
Unassigned Type           float64
Total                     float64
Year                        int64
dtype: object


In [11]:
df.head()

,Date,Spacecraft,Rocket Bodies,Mission-Related Debris,Fragmentation Debris,Unassigned Type,Total,Year
0,1957.00,0.0,0.0,0.0,0.0,0.0,0.0,1957
1,1957.09,0.0,0.0,0.0,0.0,0.0,0.0,1957
2,1957.16,0.0,0.0,0.0,0.0,0.0,0.0,1957
3,1957.25,0.0,0.0,0.0,0.0,0.0,0.0,1957
4,1957.33,0.0,0.0,0.0,0.0,0.0,0.0,1957


In [12]:
df.tail()

,Date,Spacecraft,Rocket Bodies,Mission-Related Debris,Fragmentation Debris,Unassigned Type,Total,Year
811,2024.58,13032.0,2128.0,1888.0,12064.0,563.0,29675.0,2024
812,2024.67,13046.0,2124.0,1895.0,12015.0,572.0,29652.0,2024
813,2024.75,13151.0,2129.0,1890.0,11972.0,563.0,29705.0,2024
814,2024.84,13357.0,2137.0,1889.0,11900.0,629.0,29912.0,2024
815,2024.92,13449.0,2136.0,1877.0,11791.0,621.0,29874.0,2024


In [13]:
df["Debris"] = (
    df["Rocket Bodies"]
    + df["Mission-Related Debris"]
    + df["Fragmentation Debris"]
    + df["Unassigned Type"]
)

In [15]:
print(df.shape)
print(df[["Date", "Total", "Debris"]].tail(10))

(816, 9)
        Date    Total   Debris
806  2024.17  28528.0  16245.0
807  2024.25  28637.0  16192.0
808  2024.33  28762.0  16122.0
809  2024.42  28831.0  16079.0
810  2024.50  28793.0  16009.0
811  2024.58  29675.0  16643.0
812  2024.67  29652.0  16606.0
813  2024.75  29705.0  16554.0
814  2024.84  29912.0  16555.0
815  2024.92  29874.0  16425.0


In [16]:
df = df.sort_values("Date").reset_index(drop=True)

In [17]:
df["Debris_lag_1"] = df["Debris"].shift(1)
df["Debris_lag_2"] = df["Debris"].shift(2)
df["Debris_lag_3"] = df["Debris"].shift(3)
df["Debris_lag_6"] = df["Debris"].shift(6)
df["Debris_lag_12"] = df["Debris"].shift(12)

In [18]:
df["Growth"] = df["Debris"].diff()

df["Growth_lag_1"] = df["Growth"].shift(1)
df["Growth_lag_3"] = df["Growth"].shift(3)
df["Growth_lag_12"] = df["Growth"].shift(12)

In [19]:
df["Time"] = np.arange(len(df))

In [20]:
df = df.dropna().reset_index(drop=True)

In [21]:
print(df.shape)

(803, 19)


In [22]:
df[
    [
        "Date",
        "Debris",
        "Debris_lag_1",
        "Debris_lag_3",
        "Debris_lag_12",
        "Growth",
        "Growth_lag_1",
        "Time"
    ]
].head(10)

,Date,Debris,Debris_lag_1,Debris_lag_3,Debris_lag_12,Growth,Growth_lag_1,Time
0,1958.09,0.0,0.0,1.0,0.0,0.0,-1.0,13
1,1958.16,2.0,0.0,1.0,0.0,2.0,0.0,14
2,1958.25,2.0,2.0,0.0,0.0,0.0,2.0,15
3,1958.33,3.0,2.0,0.0,0.0,1.0,0.0,16
4,1958.42,3.0,3.0,2.0,0.0,0.0,1.0,17
5,1958.50,3.0,3.0,2.0,0.0,0.0,0.0,18
6,1958.58,3.0,3.0,3.0,0.0,0.0,0.0,19
7,1958.67,3.0,3.0,3.0,0.0,0.0,0.0,20
8,1958.75,3.0,3.0,3.0,1.0,0.0,0.0,21
9,1958.84,3.0,3.0,3.0,1.0,0.0,0.0,22


In [23]:
target = "Debris"

predictors = [
    "Debris_lag_1",
    "Debris_lag_2",
    "Debris_lag_3",
    "Debris_lag_6",
    "Debris_lag_12",
    "Growth_lag_1",
    "Growth_lag_3",
    "Growth_lag_12",
    "Time"
]

In [24]:
split = int(len(df) * 0.8)

train = df.iloc[:split].copy()
test = df.iloc[split:].copy()

X_train = train[predictors]
y_train = train[target]

X_test = test[predictors]
y_test = test[target]

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Training period:", train["Date"].min(), "to", train["Date"].max())
print("Testing period:", test["Date"].min(), "to", test["Date"].max())

Train: (642, 9)
Test: (161, 9)
Training period: 1958.09 to 2011.5
Testing period: 2011.58 to 2024.92


In [25]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)

ridge.fit(X_train, y_train)

ridge_pred = ridge.predict(X_test)


In [26]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, ridge_pred)
rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
mape = np.mean(np.abs((y_test - ridge_pred) / y_test)) * 100
r2 = r2_score(y_test, ridge_pred)

print("Ridge Results")
print("MAE :", mae)
print("RMSE:", rmse)
print("MAPE:", mape, "%")
print("R²  :", r2)

Ridge Results
MAE : 78.70682211176832
RMSE: 179.32638850496826
MAPE: 0.4889012475509146 %
R²  : 0.9639355247546031


In [27]:
from sklearn.ensemble import RandomForestRegressor

In [28]:
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

In [29]:
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mape = np.mean(np.abs((y_test - rf_pred) / y_test)) * 100
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Results")
print("MAE :", rf_mae)
print("RMSE:", rf_rmse)
print("MAPE:", rf_mape, "%")
print("R²  :", rf_r2)

Random Forest Results
MAE : 742.5841282847582
RMSE: 957.8637696966697
MAPE: 4.649890067420694 %
R²  : -0.028960507711676575
